# Model Training Notebook

This is the Jupyter notebook where both the cross-validation and final training + evaluation were run. The snippets in this notebook call other modules in this directory that did the heavy lifting/automation for these steps (e.g. [keras_utility_classes.py](keras_utility_classes.py)). There are analogous notebooks for the other two model architectures, which facilitated training all three in parallel using multiple systems. **There are two important things to note about these notebooks:**

+ During the project there was an issue with the code for my initial cross-validation which led to some metrics not being logged properly. As such, even though I had completed cross-validation, I did not have a few of the metrics I wanted to put into the plots in the manuscript/final report (i.e. supplemental Figure 13). I re-ran cross-validation towards the end of the project on the same CV folds to get some of these metrics for the report. This is why the dates/timestamps for cross-validation in these files are later than the those of the final training run.

+ A few cell outputs in these notebooks stop mid-epoch. This was due to a bug where the Jupyter notebook would occasionally lose communication with the main process and stop updating, despite the fact that training continued successfully to the end (as indicated by fully completed logs of all epochs and corresponding model weights).

### Make and save the folds used for cross-validation comparison of models

In [ ]:
# import keras_models as my_models
# import keras_utility_classes as kutils
# from preparation_1 import structural_profiles
# from get_normalization_params_5 import load_json

# # Load the estimated class weights
# class_weights = f'{structural_profiles}class_weights.json'
# class_weights = load_json(class_weights, verbose=True)

# # # Initialize a trainer object
# trainer = kutils.TrainingRun(class_weights = class_weights,
#                              class_encoding = {'control_exons' : 0, 
#                                                'control_introns_intergenic' : 0, 
#                                                'control_introns_intron' : 0, 
#                                                'intron-exon' : 1, 
#                                                'exon-intron' : 2})

# trainer.get_CV_folds(N_folds=5, save_dir='./CV_folds/')

# MBDA-Net cross validation

In [2]:
import keras_models as my_models

model_init ={'filters' : 64, 
             'kernel_sizes' : [4, 8, 16, 32], 
             'strides' : 1, 
             'mem_units' : 64, 
             'final_dense' : 64,
             'bidirectional' : True}

model = my_models.MBDA_Net((77,28), 3, **model_init, print_summary=True)

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 77, 28)    │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_8 (Conv1D)   │ (None, 77, 64)    │     57,408 │ input_layer_1[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_9 (Conv1D)   │ (None, 77, 64)    │     28,736 │ input_layer_1[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 77, 64)    │        256 │ conv1d_8[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 77, 64)    │        256 │ conv1d_9[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_4 (ReLU)      │ (None, 77, 64)    │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_5 (ReLU)      │ (None, 77, 64)    │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_4       │ (None, 77, 128)   │          0 │ re_lu_4[0][0],    │
│ (Concatenate)       │                   │            │ re_lu_5[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_10 (Conv1D)  │ (None, 77, 64)    │     14,400 │ input_layer_1[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ attention_3         │ (None, 77, 128)   │          0 │ concatenate_4[0]… │
│ (Attention)         │                   │            │ concatenate_4[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 77, 64)    │        256 │ conv1d_10[0][0]   │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_12 (Conv1D)  │ (None, 77, 64)    │      8,256 │ attention_3[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_6 (ReLU)      │ (None, 77, 64)    │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_5       │ (None, 77, 128)   │          0 │ conv1d_12[0][0],  │
│ (Concatenate)       │                   │            │ re_lu_6[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_11 (Conv1D)  │ (None, 77, 64)    │      7,232 │ input_layer_1[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ attention_4         │ (None, 77, 128)   │          0 │ concatenate_5[0]… │
│ (Attention)         │                   │            │ concatenate_5[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 77, 64)    │        256 │ conv1d_11[0][0]   │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_13 (Conv1D)  │ (None, 77, 64)    │      8,256 │ attention_4[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_7 (ReLU)      │ (None, 77, 64)    │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_6       │ (None, 77, 128)   │          0 │ conv1d_13[0][0],  │
│ (Concatenate)       │                   │            │ re_lu_7[0][0]   

 Total params: 534,467 (2.04 MB)

 Trainable params: 533,827 (2.04 MB)

 Non-trainable params: 640 (2.50 KB)

In [ ]:
import keras_models as my_models
import keras_utility_classes as kutils
from preparation_1 import structural_profiles
from get_normalization_params_5 import load_json

# Load the estimated class weights
class_weights = f'{structural_profiles}class_weights.json'
class_weights = load_json(class_weights, verbose=True)

# Define the folder where training data will be stored
project_folder = './MBDA-Net_best_cv/'
base_name = 'MBDA-Net' # No underscores!
kutils.check_make_dir(project_folder)

# # Initialize a trainer object
trainer = kutils.TrainingRun(class_weights = class_weights,
                             class_encoding = {'control_exons' : 0, 
                                               'control_introns_intergenic' : 0, 
                                               'control_introns_intron' : 0, 
                                               'intron-exon' : 1, 
                                               'exon-intron' : 2})

# Name constructor function for the model type to use (one found in keras_models) and define required hyperparameters
model_creator = my_models.MBDA_Net
model_init ={'filters' : 64, 
             'kernel_sizes' : [4, 8, 16, 32], 
             'strides' : 1, 
             'mem_units' : 64, 
             'final_dense' : 64,
             'bidirectional' : True,
             'print_summary' : True}

# Run the model training in 5-fold cross validation mode, saving results and best models from each run
trainer.execute_training_run(model_creator=model_creator, 
                             model_args=model_init,
                             saves_base_dir=project_folder,
                             base_name=base_name,
                             batch_size=64, # 64 was ~30 per epoch
                             max_epochs=10,
                             patience=0, 
                             run_type='cross_validation',
                             N_folds=5, 
                             folds_dir='./CV_folds/',
                             lr_decay=False, # Last stint took ??
                             lr=0.0002, # 1/5 default learning rate # LATEST: 1274m for ~7 epochs to converge/stop 28.3 hours estimated for final train
                             plot_results=False) # 9078 s/epoch ~6 to converge; 10553s/epoch with all metrics on, 14-17 hrs for one CV fold! (22 hrs on full train set)

2025-10-10 18:51:22.880512: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-10-10 18:51:23.694068: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-10-10 18:51:25.941238: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


load_json: Loaded ./3_Physicochemical_Profiles/class_weights.json.
TrainingRun(): Loading neccesary .json and .npy data files...
write_json_pretty: Wrote ./MBDA-Net_best_cv/training_call.json.
execute_training_run(): loading cross-validation splits from files in ./CV_folds/...
execute_training_run(): it appears that 3/5 have already been completed. Resuming on fold 4.


I0000 00:00:1760136745.217897     959 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 5564 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3070, pci bus id: 0000:65:00.0, compute capability: 8.6


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 77, 28)    │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d (Conv1D)     │ (None, 77, 64)    │     57,408 │ input_layer[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_1 (Conv1D)   │ (None, 77, 64)    │     28,736 │ input_layer[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 77, 64)    │        256 │ conv1d[0][0]      │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 77, 64)    │        256 │ conv1d_1[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu (ReLU)        │ (None, 77, 64)    │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_1 (ReLU)      │ (None, 77, 64)    │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 77, 128)   │          0 │ re_lu[0][0],      │
│ (Concatenate)       │                   │            │ re_lu_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_2 (Conv1D)   │ (None, 77, 64)    │     14,400 │ input_layer[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ attention           │ (None, 77, 128)   │          0 │ concatenate[0][0… │
│ (Attention)         │                   │            │ concatenate[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 77, 64)    │        256 │ conv1d_2[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_4 (Conv1D)   │ (None, 77, 64)    │      8,256 │ attention[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_2 (ReLU)      │ (None, 77, 64)    │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_1       │ (None, 77, 128)   │          0 │ conv1d_4[0][0],   │
│ (Concatenate)       │                   │            │ re_lu_2[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_3 (Conv1D)   │ (None, 77, 64)    │      7,232 │ input_layer[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ attention_1         │ (None, 77, 128)   │          0 │ concatenate_1[0]… │
│ (Attention)         │                   │            │ concatenate_1[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 77, 64)    │        256 │ conv1d_3[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_5 (Conv1D)   │ (None, 77, 64)    │      8,256 │ attention_1[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_3 (ReLU)      │ (None, 77, 64)    │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_2       │ (None, 77, 128)   │          0 │ conv1d_5[0][0],   │
│ (Concatenate)       │                   │            │ re_lu_3[0][0]   

...

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 77, 28)    │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_8 (Conv1D)   │ (None, 77, 64)    │     57,408 │ input_layer_1[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_9 (Conv1D)   │ (None, 77, 64)    │     28,736 │ input_layer_1[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 77, 64)    │        256 │ conv1d_8[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 77, 64)    │        256 │ conv1d_9[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_4 (ReLU)      │ (None, 77, 64)    │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_5 (ReLU)      │ (None, 77, 64)    │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_4       │ (None, 77, 128)   │          0 │ re_lu_4[0][0],    │
│ (Concatenate)       │                   │            │ re_lu_5[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_10 (Conv1D)  │ (None, 77, 64)    │     14,400 │ input_layer_1[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ attention_3         │ (None, 77, 128)   │          0 │ concatenate_4[0]… │
│ (Attention)         │                   │            │ concatenate_4[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 77, 64)    │        256 │ conv1d_10[0][0]   │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_12 (Conv1D)  │ (None, 77, 64)    │      8,256 │ attention_3[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_6 (ReLU)      │ (None, 77, 64)    │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_5       │ (None, 77, 128)   │          0 │ conv1d_12[0][0],  │
│ (Concatenate)       │                   │            │ re_lu_6[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_11 (Conv1D)  │ (None, 77, 64)    │      7,232 │ input_layer_1[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ attention_4         │ (None, 77, 128)   │          0 │ concatenate_5[0]… │
│ (Attention)         │                   │            │ concatenate_5[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 77, 64)    │        256 │ conv1d_11[0][0]   │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_13 (Conv1D)  │ (None, 77, 64)    │      8,256 │ attention_4[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_7 (ReLU)      │ (None, 77, 64)    │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_6       │ (None, 77, 128)   │          0 │ conv1d_13[0][0],  │
│ (Concatenate)       │                   │            │ re_lu_7[0][0]   

 Total params: 534,467 (2.04 MB)

 Trainable params: 533,827 (2.04 MB)

 Non-trainable params: 640 (2.50 KB)

execute_training_run(): fold 5/5 training will now commence!
Epoch 1/10
55984/55984 ━━━━━━━━━━━━━━━━━━━━ 0s 162ms/step - AUC_PR_Control: 0.9900 - AUC_PR_Exon-Intron: 0.5290 - AUC_PR_Intron-Exon: 0.6507 - AUC_ROC_Control: 0.8973 - AUC_ROC_Exon-Intron: 0.8748 - AUC_ROC_Intron-Exon: 0.9265 - F1_Score_0.1: 0.6963 - F1_Score_0.2: 0.6634 - F1_Score_0.3: 0.6322 - F1_Score_0.4: 0.6026 - F1_Score_0.5: 0.5738 - F1_Score_0.6: 0.5437 - F1_Score_0.7: 0.5101 - F1_Score_0.8: 0.4699 - F1_Score_0.9: 0.4132 - F1_Score_argmax: 0.5745 - FPR_Control: 0.6610 - FPR_Exon-Intron: 0.0201 - FPR_Intron-Exon: 0.0148 - Precision_Control: 0.9635 - Precision_Exon-Intron: 0.7425 - Precision_Intron-Exon: 0.7634 - Recall_Control: 0.9636 - Recall_Exon-Intron: 0.2458 - Recall_Intron-Exon: 0.3082 - Recall_at_90P_Control: 1.0000 - Recall_at_90P_Exon-Intron: 0.2680 - Recall_at_90P_Intron-Exon: 0.3642 - categorical_accuracy: 0.9585 - kl_divergence: 0.1554 - loss: 0.3294
Epoch 1: val_loss improved from None to 0.15869, saving 

# MBDA-Net Final training!

In [ ]:
import keras_models as my_models
import keras_utility_classes as kutils
from preparation_1 import structural_profiles
from get_normalization_params_5 import load_json

# Load the estimated class weights
class_weights = f'{structural_profiles}class_weights.json'
class_weights = load_json(class_weights, verbose=True)

# Define the folder where training data will be stored
project_folder = './MBDA-Net_final/'
base_name = 'MBDA-net' # No underscores!
kutils.check_make_dir(project_folder)

# # Initialize a trainer object
trainer = kutils.TrainingRun(class_weights = class_weights,
                             class_encoding = {'control_exons' : 0, 
                                               'control_introns_intergenic' : 0, 
                                               'control_introns_intron' : 0, 
                                               'intron-exon' : 1, 
                                               'exon-intron' : 2})

# Name constructor function for the model type to use (one found in keras_models) and define required hyperparameters
model_creator = my_models.MBDA_Net
model_init ={'filters' : 64, 
             'kernel_sizes' : [4, 8, 16, 32], 
             'strides' : 1, 
             'mem_units' : 64, 
             'final_dense' : 64,
             'bidirectional' : True,
             'print_summary' : True}

# Run the model training in 5-fold cross validation mode, saving results and best models from each run
trainer.execute_training_run(model_creator=model_creator, 
                             model_args=model_init,
                             saves_base_dir=project_folder,
                             base_name=base_name,
                             batch_size=64,
                             max_epochs=10,
                             patience=0, 
                             run_type='final_eval',
                             lr_decay=False, # Full train: 1190 minutes # Forreal 1023 min for final train
                             lr=0.0002, # 1/5 default learning rate # LATEST: 1274m for ~7 epochs to converge/stop 28.3 hours estimated for final train
                             plot_results=False) # 9078 s/epoch ~6 to converge; 10553s/epoch with all metrics on, 14-17 hrs for one CV fold! (22 hrs on full train set)

load_json: Loaded ./3_Physicochemical_Profiles/class_weights.json.
check_make_dir(): made ./MBDA-Net_final/.
TrainingRun(): Loading neccesary .json and .npy data files...
write_json_pretty: Wrote ./MBDA-Net_final/training_call.json.


Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_2       │ (None, 77, 28)    │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_16 (Conv1D)  │ (None, 77, 64)    │     57,408 │ input_layer_2[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_17 (Conv1D)  │ (None, 77, 64)    │     28,736 │ input_layer_2[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 77, 64)    │        256 │ conv1d_16[0][0]   │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 77, 64)    │        256 │ conv1d_17[0][0]   │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_8 (ReLU)      │ (None, 77, 64)    │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_9 (ReLU)      │ (None, 77, 64)    │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_8       │ (None, 77, 128)   │          0 │ re_lu_8[0][0],    │
│ (Concatenate)       │                   │            │ re_lu_9[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_18 (Conv1D)  │ (None, 77, 64)    │     14,400 │ input_layer_2[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ attention_6         │ (None, 77, 128)   │          0 │ concatenate_8[0]… │
│ (Attention)         │                   │            │ concatenate_8[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 77, 64)    │        256 │ conv1d_18[0][0]   │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_20 (Conv1D)  │ (None, 77, 64)    │      8,256 │ attention_6[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_10 (ReLU)     │ (None, 77, 64)    │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_9       │ (None, 77, 128)   │          0 │ conv1d_20[0][0],  │
│ (Concatenate)       │                   │            │ re_lu_10[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_19 (Conv1D)  │ (None, 77, 64)    │      7,232 │ input_layer_2[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ attention_7         │ (None, 77, 128)   │          0 │ concatenate_9[0]… │
│ (Attention)         │                   │            │ concatenate_9[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 77, 64)    │        256 │ conv1d_19[0][0]   │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_21 (Conv1D)  │ (None, 77, 64)    │      8,256 │ attention_7[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_11 (ReLU)     │ (None, 77, 64)    │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_10      │ (None, 77, 128)   │          0 │ conv1d_21[0][0],  │
│ (Concatenate)       │                   │            │ re_lu_11[0][0]  

 Total params: 534,467 (2.04 MB)

 Trainable params: 533,827 (2.04 MB)

 Non-trainable params: 640 (2.50 KB)

execute_training_run(): final training and evaluation run will soon commence!
execute_training_run(): training will now commence!
Epoch 1/10


2025-09-27 19:06:34.618853: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:473] Loaded cuDNN version 91300


69981/69981 ━━━━━━━━━━━━━━━━━━━━ 0s 160ms/step - AUC_PR_Control: 0.9913 - AUC_PR_Exon-Intron: 0.5876 - AUC_PR_Intron-Exon: 0.7053 - AUC_ROC_Control: 0.9111 - AUC_ROC_Exon-Intron: 0.8918 - AUC_ROC_Intron-Exon: 0.9356 - F1_Score_0.1: 0.7315 - F1_Score_0.2: 0.7061 - F1_Score_0.3: 0.6791 - F1_Score_0.4: 0.6517 - F1_Score_0.5: 0.6241 - F1_Score_0.6: 0.5941 - F1_Score_0.7: 0.5589 - F1_Score_0.8: 0.5133 - F1_Score_0.9: 0.4447 - F1_Score_argmax: 0.6244 - FPR_Control: 0.6192 - FPR_Exon-Intron: 0.0186 - FPR_Intron-Exon: 0.0130 - Precision_Control: 0.9655 - Precision_Exon-Intron: 0.7533 - Precision_Intron-Exon: 0.7823 - Recall_Control: 0.9665 - Recall_Exon-Intron: 0.2931 - Recall_Intron-Exon: 0.3442 - Recall_at_90P_Control: 1.0000 - Recall_at_90P_Exon-Intron: 0.3396 - Recall_at_90P_Intron-Exon: 0.4490 - categorical_accuracy: 0.9610 - kl_divergence: 0.1423 - loss: 0.3101
Epoch 1: val_loss improved from None to 0.15399, saving model to ./MBDA-Net_final/MBDA-net_best.keras
69981/69981 ━━━━━━━━━━━━━━

# Save the best model from the final training

In [ ]:
import keras_models as my_models

model_init ={'filters' : 64, 
             'kernel_sizes' : [4, 8, 16, 32], 
             'strides' : 1, 
             'mem_units' : 64, 
             'final_dense' : 64,
             'bidirectional' : True}

model = my_models.MBDA_Net((77,28), 3, **model_init, print_summary=True)